In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                        NORMALIZAÇÃO DE DADOS                                 ║
# ║          Padronização, Auditoria e Exportação das Features                   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  IDENTIFICAÇÃO
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Autor       : Yago
#  Instituição : Universidade Federal do Ceará
#  Curso       : Engenharia Elétrica
#  Versão      : 3.0
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  DESCRIÇÃO / FUNÇÃO DO SCRIPT
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Este script implementa a etapa de normalização de dados da arquitetura do
#  TCC, aplicada ao submercado Nordeste (PLD NORDESTE — Cenário A).
#  Realiza a padronização das features selecionadas na etapa anterior (seleção
#  de variáveis) com garantia anti-leakage rigorosa: o RobustScaler é fitado
#  exclusivamente no conjunto de treino e aplicado (transform) tanto no treino
#  quanto no teste com os mesmos parâmetros — o teste nunca influencia o fit.
#
#  O script processa apenas o Cenário A (todos os anos do treino), conforme
#  definido em SCENARIOS. O Cenário B (sem 2022–2023) foi removido nesta versão.
#
#  Ao final da execução, o script gera por cenário + estratégia:
#    (a) arquivos Parquet normalizados de treino e teste;
#    (b) scaler serializado em .pkl e parâmetros em .json;
#    (c) CSV de auditoria com estatísticas antes e depois da normalização;
#    (d) figuras de distribuição (antes × depois) e boxplot das features;
#    (e) relatório consolidado em CSV e tabelas Rich no terminal.
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  ENTRADAS (INPUTS)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  O script consome os arquivos gerados pela etapa de seleção de variáveis.
#  Os parâmetros de controle estão definidos nas constantes abaixo:
#
#    INPUT_DIR  : BASE_DIR / "9_selecao_variaveis"
#    SCENARIOS  : lista de cenários a processar (padrão: apenas Cenário A)
#    STRATEGIES : lista de estratégias de seleção (padrão: HYBRID_AGGRESSIVE)
#
#  Estrutura esperada de entrada (por cenário + estratégia):
#    INPUT_DIR / {cenario} / {strategy} / TREINO_SELECTED.parquet
#    INPUT_DIR / {cenario} / {strategy} / TESTE_SELECTED.parquet
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  SAÍDAS (OUTPUTS)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Por cenário + estratégia em SAVE_DIR / {cenario} / {strategy} /:
#
#  1. TREINO_NORM.parquet       — features normalizadas do conjunto de treino
#  2. TESTE_NORM.parquet        — features normalizadas do conjunto de teste
#  3. scaler.pkl                — objeto RobustScaler serializado
#  4. scaler_params.json        — mediana (center_) e IQR (scale_) por feature
#  5. auditoria_stats_antes_depois.csv
#                               — estatísticas descritivas antes e após norm.
#  6. figuras/
#     ├── distribuicao_antes_depois_top20_{strategy}.png
#     └── boxplot_features_normalizadas_top20_{strategy}.png
#
#  Relatório geral:
#  7. SAVE_DIR / relatorio_normalizacao_nordeste.csv
#
#  Saída no terminal (Rich):
#     • Painel de boas-vindas com arquitetura anti-leakage
#     • Tabela de parâmetros do RobustScaler por feature
#     • Painel de resumo por cenário + estratégia
#     • Relatório de tempo por estágio
#     • Tabela consolidada de todos os cenários processados
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  CENÁRIO PROCESSADO
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Rótulo                    Descrição
#  cenario_A_todos_anos      FIT com todos os anos disponíveis no treino
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  CONSIDERAÇÕES INICIAIS E OBSERVAÇÕES TÉCNICAS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
#  1. AMBIENTE DE EXECUÇÃO
#     O script foi desenvolvido e testado no Google Colab (Python 3.10+).
#     Pode ser executado localmente, mas requer ajuste manual de BASE_DIR.
#     Antes de executar, montar o Google Drive:
#
#         from google.colab import drive
#         drive.mount('/content/drive')
#
#  2. PRÉ-REQUISITO
#     Este script depende dos arquivos .parquet gerados pela etapa de seleção
#     de variáveis (passo 9). Certifique-se de que essa etapa foi concluída
#     com sucesso antes de executar este script.
#
#  3. ARQUITETURA ANTI-LEAKAGE
#     A garantia anti-leakage é o princípio central deste script:
#       • FIT   do RobustScaler → SOMENTE no TREINO de cada cenário
#       • TRANSFORM             → TREINO + TESTE com os mesmos parâmetros
#       • TARGET                → NÃO normalizado (escala original preservada)
#       • KEY_* / binárias      → NÃO normalizadas (passthrough direto)
#     Cada cenário possui seu próprio scaler independente, garantindo
#     que nenhuma informação do teste vaze para o processo de fit.
#
#  4. CORREÇÕES APLICADAS (v3.0)
#     • Removido o Cenário B (sem_2022_2023) — roda apenas o Cenário A.
#     • Silenciado pandas.errors.PerformanceWarning de DataFrame fragmentado,
#       disparado em _fit_transform() ao inserir colunas normalizadas uma a uma.
#       Trata-se de aviso de performance apenas — não afeta o resultado.
#
#  5. ESCOLHA DO ROBUSTSCALER
#     O RobustScaler foi escolhido por sua resistência a outliers extremos,
#     comuns em séries de PLD (spikes de preço). Utiliza mediana (center_) e
#     IQR (scale_) calculados no intervalo [5%, 95%] para cada feature,
#     ignorando os 5% mais extremos de cada extremo (quantile_range = (5, 95)).
#
#  6. COLUNAS PASSTHROUGH (NÃO NORMALIZADAS)
#     As seguintes categorias de colunas passam direto sem normalização:
#       • Prefixo KEY_*          : colunas-chave temporais e de submercado
#       • Flags binárias         : IS_FIM_DE_SEMANA, IS_FERIADO, SPIKE, etc.
#       • Regimes                : REGIME_VOL, REGIME_PRECO, REGIME_CMO
#       • Cíclicas               : HORA_SEN, HORA_COS, MES_SEN, MES_COS (já em [-1,1])
#       • Granularidades pequenas: DIA_SEMANA, HORA
#
#  7. GERENCIAMENTO DE MEMÓRIA
#     Arrays intermediários (X_treino_num, X_treino_norm, etc.) são
#     explicitamente deletados e gc.collect() é chamado após cada estágio
#     relevante. O monitor de RAM imprime alertas em amarelo (≥65%) e
#     vermelho (≥80%), executando gc.collect() preventivo no nível crítico.
#
#  8. DETECÇÃO DINÂMICA DE TARGET
#     A função detectar_target() tenta localizar a coluna-alvo em três níveis:
#       1. Nome exato (TARGET_CANONICAL)
#       2. Busca por tokens ["nordeste", "target"] no nome da coluna
#       3. Fallback com pares de tokens alternativos (["target","nordeste"],
#          ["pld","nordeste"])
#     Se nenhuma correspondência for encontrada, levanta KeyError com
#     diagnóstico das 30 primeiras colunas disponíveis.
#
#  9. REPRODUTIBILIDADE
#     Os parâmetros exatos do scaler (mediana e IQR por feature) são
#     persistidos em scaler_params.json junto com metadados do run
#     (timestamp, versão, quantile_range), garantindo reprodutibilidade
#     total da etapa de transformação em inferência futura.
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  DEPENDÊNCIAS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Biblioteca        Versão mínima  Finalidade
#  numpy             1.23           Operações numéricas e cast de arrays
#  pandas            1.5            Manipulação de DataFrames e Parquet
#  scikit-learn      1.1            RobustScaler (normalização robusta)
#  matplotlib        3.5            Geração das figuras de distribuição
#  psutil            5.9            Monitoramento de uso de RAM
#  rich              13.0           Tabelas, painéis e progresso no terminal
#  pickle            —              Serialização do objeto scaler
#  json              —              Exportação dos parâmetros do scaler
#
#  Instalação rápida (Google Colab / pip):
#      !pip install rich psutil scikit-learn matplotlib
#  (numpy, pandas e demais são stdlib ou já disponíveis no Colab por padrão)
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  ESTRUTURA DE DIRETÓRIOS GERADA
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  10_dados_normalizados/
#  ├── cenario_A_todos_anos/
#  │   └── HYBRID_AGGRESSIVE/
#  │       ├── TREINO_NORM.parquet
#  │       ├── TESTE_NORM.parquet
#  │       ├── scaler.pkl
#  │       ├── scaler_params.json
#  │       ├── auditoria_stats_antes_depois.csv
#  │       └── figuras/
#  │           ├── distribuicao_antes_depois_top20_HYBRID_AGGRESSIVE.png
#  │           └── boxplot_features_normalizadas_top20_HYBRID_AGGRESSIVE.png
#  └── relatorio_normalizacao_nordeste.csv
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  COMO EXECUTAR (Google Colab)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Célula 1 — Montar o Drive:
#      from google.colab import drive
#      drive.mount('/content/drive')
#
#  Célula 2 — Instalar dependências:
#      !pip install rich psutil scikit-learn matplotlib
#
#  Célula 3 — Executar o script:
#      exec(open('normalizacao_pld_nordeste.py').read())
#   ou simplesmente executar este arquivo como módulo principal.
#
# ══════════════════════════════════════════════════════════════════════════════

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

import gc
import json
import pickle
import re
import time
from contextlib import contextmanager
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
from sklearn.preprocessing import RobustScaler

# FIX v3.0: silencia o PerformanceWarning de DataFrame fragmentado.
# Precisa vir DEPOIS do import do pandas (pd.errors só existe após o import).
# Esse aviso é apenas de performance — disparava em _fit_transform() ao inserir
# colunas normalizadas uma a uma no DataFrame de saída. Não é erro e não afeta
# o resultado.
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

from rich import box
from rich.console import Console
from rich.panel import Panel
from rich.progress import BarColumn, Progress, SpinnerColumn, TextColumn, TimeElapsedColumn
from rich.rule import Rule
from rich.table import Table
from rich.text import Text
from rich.theme import Theme


# ==============================================================================
# 🎨 TEMA VISUAL
# ==============================================================================
THEME = Theme({
    "info"      : "bold cyan",
    "warning"   : "bold yellow",
    "error"     : "bold red",
    "success"   : "bold green",
    "header"    : "bold white on dark_blue",
    "highlight" : "bold magenta",
    "muted"     : "dim white",
    "skip"      : "dim yellow",
    "norm"      : "bold cyan",
    "passthru"  : "dim white",
    "ram_ok"    : "bold green",
    "ram_warn"  : "bold yellow",
    "ram_crit"  : "bold red",
    "cenario_a" : "bold green",
    "cenario_b" : "bold magenta",
})
console = Console(theme=THEME)


# ==============================================================================
# ⚙️ CONFIGURAÇÕES DO AMBIENTE
# ==============================================================================
BASE_DIR  = Path("/content/drive/MyDrive/TCC_PLD_Project/09-ESCRITA_TCC/PARTES_TCC/codigos/dados")
INPUT_DIR = BASE_DIR / "9_selecao_variaveis"
SAVE_DIR  = BASE_DIR / "10_dados_normalizados"

# ── Cenários a processar ──────────────────────────────────────────────────────
# FIX v3.0: apenas o Cenário A (todos os anos do treino).
# O Cenário B (sem_2022_2023) foi removido nesta versão.
SCENARIOS: List[Dict] = [
    {
        "label" : "cenario_A_todos_anos",
        "desc"  : "FIT com todos os anos do treino",
        "cor"   : "cenario_a",
        "border": "green",
    },
]

# ── Estratégias de seleção de variáveis ──────────────────────────────────────
STRATEGIES: List[str] = ["HYBRID_AGGRESSIVE"]

# ── Coluna-alvo (TARGET) do submercado Nordeste ───────────────────────────────
TARGET_CANONICAL : str       = "01_CCEE_NORDESTE_TARGET_PLD_HORA_NORDESTE"
_TARGET_TOKENS   : List[str] = ["nordeste", "target"]

# ── Colunas que NÃO devem ser normalizadas (passthrough) ─────────────────────
# Prefixos: colunas que comecem com qualquer um desses strings são excluídas do fit
PASSTHROUGH_PREFIXES: Tuple[str, ...] = ("KEY_",)

# Nomes exatos: flags binárias, regimes, granularidades e features cíclicas
PASSTHROUGH_EXACT: Tuple[str, ...] = (
    # Flags binárias de calendário e operação
    "IS_FIM_DE_SEMANA", "IS_FERIADO", "IS_DIA_UTIL",
    "SPIKE", "SPIKE_ALTO", "SPIKE_BAIXO", "HORARIO_PICO", "ALERTA_ESCASSEZ",
    # Regimes categóricos ordinais (não têm escala contínua)
    "REGIME_VOL", "REGIME_PRECO", "REGIME_CMO",
    # Granularidades pequenas (não se beneficiam de normalização)
    "DIA_SEMANA", "HORA",
    # Features cíclicas — já estão na escala [-1, 1]
    "HORA_SEN", "HORA_COS", "MES_SEN", "MES_COS",
)

# ── Parâmetros de normalização e geração de figuras ──────────────────────────
NORM_CONFIG: Dict = {
    "quantile_range" : (5.0, 95.0),   # ignora 5% de cada extremo — robusto a spikes de PLD
    "fig_top_n"      : 20,             # top-N features exibidas nas figuras
    "fig_dpi"        : 120,            # resolução das figuras exportadas
    "ram_warn_pct"   : 65,             # % de RAM para alerta amarelo
    "ram_crit_pct"   : 80,             # % de RAM para alerta vermelho + gc preventivo
    "dtype_chunk"    : 100,            # colunas por chunk no cast de dtype
}


# ==============================================================================
# 🎯 DETECÇÃO DINÂMICA DE TARGET
# ==============================================================================
def detectar_target(df: pd.DataFrame) -> str:
    """
    Localiza a coluna-alvo (TARGET) no DataFrame em três níveis:
      1. Nome exato (TARGET_CANONICAL)
      2. Busca por tokens _TARGET_TOKENS no nome da coluna (case-insensitive)
      3. Fallback com pares alternativos de tokens
    Levanta KeyError com diagnóstico se nenhuma correspondência for encontrada.
    """
    if TARGET_CANONICAL in df.columns:
        console.print(f"   [success]✔  TARGET (nome exato): [bold]{TARGET_CANONICAL}[/][/]")
        return TARGET_CANONICAL

    cols_lower = {c: c.lower() for c in df.columns}

    candidatas = [c for c, cl in cols_lower.items()
                  if all(tok in cl for tok in _TARGET_TOKENS)]
    if candidatas:
        console.print(f"   [warning]⚠  TARGET por tokens {_TARGET_TOKENS}: [bold]{candidatas[0]}[/][/]")
        return candidatas[0]

    for tokens in [["target", "nordeste"], ["pld", "nordeste"]]:
        cands = [c for c, cl in cols_lower.items() if all(t in cl for t in tokens)]
        if cands:
            console.print(f"   [warning]⚠  TARGET fallback {tokens}: [bold]{cands[0]}[/][/]")
            return cands[0]

    raise KeyError(
        f"TARGET não encontrado.\n"
        f"Canônico esperado: {TARGET_CANONICAL!r}\n"
        f"Colunas (30 primeiras): {list(df.columns[:30])}"
    )


def _is_passthrough(col: str) -> bool:
    """
    Retorna True para colunas que NÃO devem ser normalizadas.
    Verifica prefixos (KEY_*) e nomes exatos definidos em PASSTHROUGH_EXACT.
    """
    if any(col.startswith(pfx) for pfx in PASSTHROUGH_PREFIXES):
        return True
    return col in PASSTHROUGH_EXACT


# ==============================================================================
# ⏱️ UTILITÁRIO DE TEMPO
# ==============================================================================
@dataclass
class StageTimer:
    """
    Cronômetro por estágio com registro acumulado.
    Uso via context manager: with timer.track("rótulo"): ...
    """
    records: Dict[str, float] = field(default_factory=dict)
    _start : float            = field(default=0.0, init=False, repr=False)

    @contextmanager
    def track(self, label: str):
        self._start = time.perf_counter()
        try:
            yield
        finally:
            self.records[label] = round(time.perf_counter() - self._start, 2)


# ==============================================================================
# 📋 DATACLASS — Resultado por cenário + estratégia
# ==============================================================================
@dataclass
class NormResult:
    """Contêiner de resultados de uma execução de normalização."""
    cenario             : str
    strategy            : str
    target_col          : str
    cols_normalized     : List[str]
    cols_passthrough    : List[str]
    n_treino_rows       : int
    n_teste_rows        : int
    scaler_params       : Dict[str, Dict]
    treino_stats_antes  : pd.DataFrame
    treino_stats_depois : pd.DataFrame


# ==============================================================================
# 🏗️ ENGINE DE NORMALIZAÇÃO
# ==============================================================================
class NormalizationEngine:
    """
    Orquestrador do pipeline de normalização.
    Itera sobre cenários e estratégias, executando para cada combinação:
      leitura → detecção de target → fit/transform → exportação → figuras.
    """

    def __init__(
        self,
        input_dir : Path,
        save_dir  : Path,
        scenarios : List[Dict],
        strategies: List[str],
        config    : Dict,
    ) -> None:
        self.input_dir  = Path(input_dir)
        self.save_dir   = Path(save_dir)
        self.scenarios  = scenarios
        self.strategies = strategies
        self.cfg        = config
        self.timer      = StageTimer()
        self._all_results: List[Dict] = []

    # ──────────────────────────────────────────────────────────────────────────
    # 🔒 UTILITÁRIOS PRIVADOS
    # ──────────────────────────────────────────────────────────────────────────

    def _ram_status(self) -> Tuple[float, float, str]:
        """Retorna (usado_GB, total_GB, estilo_rich) com base no uso de RAM."""
        mem   = psutil.virtual_memory()
        used  = mem.used  / 1024**3
        tot   = mem.total / 1024**3
        pct   = mem.percent
        style = (
            "ram_crit" if pct >= self.cfg["ram_crit_pct"] else
            "ram_warn" if pct >= self.cfg["ram_warn_pct"] else
            "ram_ok"
        )
        return used, tot, style

    def _print_ram(self, label: str = "") -> None:
        """Imprime status de RAM com cor e executa gc preventivo se crítico."""
        used, tot, style = self._ram_status()
        pct = used / tot * 100
        tag = f" [{label}]" if label else ""
        console.print(f"   [{style}]🧠 RAM{tag}: {used:.1f}/{tot:.1f} GB ({pct:.0f}%)[/{style}]")
        if style == "ram_crit":
            console.print("   [ram_crit]⚠️  RAM CRÍTICA — gc.collect() preventivo[/]")
            gc.collect()

    def _force_free(self, *objs) -> None:
        """Deleta objetos passados e executa gc.collect() para liberar memória."""
        for o in objs:
            try:
                del o
            except Exception:
                pass
        gc.collect()

    @staticmethod
    def _clean_col_names(df: pd.DataFrame) -> pd.DataFrame:
        """Sanitiza nomes de colunas removendo caracteres especiais (", :, {, }, [, ])."""
        df.columns = [re.sub(r'[":{},\[\]]', "_", str(c)) for c in df.columns]
        return df

    # ──────────────────────────────────────────────────────────────────────────
    # 📂 CARREGAMENTO
    # ──────────────────────────────────────────────────────────────────────────

    def _load_parquet(self, path: Path, label: str) -> pd.DataFrame:
        """
        Lê arquivo Parquet, sanitiza nomes de colunas e substitui Inf/NaN por 0.0
        em todas as colunas numéricas. Processado em chunks de dtype_chunk colunas
        para evitar picos de memória em DataFrames com muitas features.
        """
        console.print(f"   [info]📂 Lendo {label}: {path.name}[/]")
        df = pd.read_parquet(path)
        df = self._clean_col_names(df)

        num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        chunk_sz = self.cfg["dtype_chunk"]
        for i in range(0, len(num_cols), chunk_sz):
            blk = num_cols[i: i + chunk_sz]
            df[blk] = (
                df[blk]
                .replace([np.inf, -np.inf], np.nan)
                .fillna(0.0)
            )

        mem_mb = df.memory_usage(deep=True).sum() / 1024**2
        console.print(f"   [info]📐 Shape {label}: {df.shape} | 💾 {mem_mb:.1f} MB[/]")
        return df

    # ──────────────────────────────────────────────────────────────────────────
    # 🔍 AUDITORIA
    # ──────────────────────────────────────────────────────────────────────────

    def _describe_cols(self, df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
        """
        Gera estatísticas descritivas (média, std, min, quartis, max) das colunas
        informadas. Limitado às primeiras 50 colunas para evitar sobrecarga.
        """
        return df[cols].describe(percentiles=[0.25, 0.5, 0.75]).T[[
            "mean", "std", "min", "25%", "50%", "75%", "max"
        ]].round(4)

    # ──────────────────────────────────────────────────────────────────────────
    # 📐 FIT + TRANSFORM — RobustScaler (anti-leakage)
    # ──────────────────────────────────────────────────────────────────────────

    def _fit_transform(
        self,
        df_treino : pd.DataFrame,
        df_teste  : pd.DataFrame,
        target_col: str,
        cenario   : str,
        strategy  : str,
    ) -> Tuple[pd.DataFrame, pd.DataFrame, RobustScaler, List[str], List[str], Dict]:
        """
        Implementa o fit/transform com garantia anti-leakage:
          • Separa colunas em dois grupos: normalizar e passthrough
          • Fita o RobustScaler SOMENTE nos dados de treino
          • Transforma treino e teste com os mesmos parâmetros
          • Reconstrói DataFrames na ordem original das colunas
          • Libera arrays intermediários após cada etapa
        Retorna: (df_treino_norm, df_teste_norm, scaler,
                  cols_normalize, cols_passthrough, scaler_params)
        """
        all_cols = df_treino.columns.tolist()

        # Separação: passthrough (target + binárias/chaves) vs. normalizar
        cols_passthrough = [c for c in all_cols if c == target_col or _is_passthrough(c)]
        cols_normalize   = [
            c for c in all_cols
            if c not in cols_passthrough
            and pd.api.types.is_numeric_dtype(df_treino[c])
        ]

        console.print(f"   [norm]📐 Colunas para normalizar   : {len(cols_normalize)}[/]")
        console.print(f"   [passthru]⏭️  Colunas passthrough (sem norm): {len(cols_passthrough)}[/]")

        # ── FIT exclusivo no TREINO (anti-leakage) ────────────────────────────
        console.print("   [info]🔧 Fit do RobustScaler no TREINO...[/]")
        scaler = RobustScaler(
            quantile_range=self.cfg["quantile_range"],
            copy=True,
        )
        X_treino_num = df_treino[cols_normalize].values.astype(np.float32)

        with self.timer.track(f"[{cenario}/{strategy}] Fit RobustScaler"):
            scaler.fit(X_treino_num)

        # Persiste parâmetros exatos por feature para reprodutibilidade
        scaler_params: Dict[str, Dict] = {}
        for i, col in enumerate(cols_normalize):
            scaler_params[col] = {
                "center_" : round(float(scaler.center_[i]), 6),
                "scale_"  : round(float(scaler.scale_[i]),  6),
                "q_range" : list(self.cfg["quantile_range"]),
            }

        # ── TRANSFORM TREINO ──────────────────────────────────────────────────
        console.print("   [info]🔄 Transform TREINO...[/]")
        with self.timer.track(f"[{cenario}/{strategy}] Transform TREINO"):
            X_treino_norm = scaler.transform(X_treino_num).astype(np.float32)

        # Reconstrói DataFrame mantendo passthrough intacto e ordem original
        df_treino_norm = df_treino[cols_passthrough].copy()
        for i, col in enumerate(cols_normalize):
            df_treino_norm[col] = X_treino_norm[:, i]
        df_treino_norm = df_treino_norm[all_cols]
        self._force_free(X_treino_num, X_treino_norm)

        # ── TRANSFORM TESTE ───────────────────────────────────────────────────
        # Usa apenas os parâmetros do fit do treino — nunca refita no teste
        console.print("   [info]🔄 Transform TESTE...[/]")
        teste_cols_norm = [c for c in cols_normalize if c in df_teste.columns]
        teste_all_cols  = [c for c in all_cols       if c in df_teste.columns]

        X_teste_num = df_teste[teste_cols_norm].values.astype(np.float32)
        with self.timer.track(f"[{cenario}/{strategy}] Transform TESTE"):
            X_teste_norm = scaler.transform(X_teste_num).astype(np.float32)

        df_teste_norm = df_teste[[c for c in cols_passthrough if c in df_teste.columns]].copy()
        for i, col in enumerate(teste_cols_norm):
            df_teste_norm[col] = X_teste_norm[:, i]
        df_teste_norm = df_teste_norm[teste_all_cols]
        self._force_free(X_teste_num, X_teste_norm)

        return (
            df_treino_norm, df_teste_norm, scaler,
            cols_normalize, cols_passthrough, scaler_params,
        )

    # ──────────────────────────────────────────────────────────────────────────
    # 📊 FIGURAS
    # ──────────────────────────────────────────────────────────────────────────

    def _save_figures(
        self,
        df_antes  : pd.DataFrame,
        df_depois : pd.DataFrame,
        cols_norm : List[str],
        target_col: str,
        fig_dir   : Path,
        cenario   : str,
        strategy  : str,
    ) -> None:
        """
        Gera e salva dois tipos de figura para as top-N features normalizadas:
          1. Histogramas antes × depois (grade N × 2)
          2. Boxplot comparativo de todas as features normalizadas
        Erros em qualquer figura são capturados e logados sem interromper o pipeline.
        """
        fig_dir.mkdir(parents=True, exist_ok=True)
        top_n = self.cfg["fig_top_n"]
        dpi   = self.cfg["fig_dpi"]
        cols  = [c for c in cols_norm if c != target_col][:top_n]
        n     = len(cols)

        # ── 1. Distribuição antes × depois ───────────────────────────────────
        try:
            fig, axes = plt.subplots(
                n, 2, figsize=(14, max(6, n * 1.8)),
                constrained_layout=True,
            )
            if n == 1:
                axes = [axes]

            for i, col in enumerate(cols):
                ax_a, ax_d = axes[i][0], axes[i][1]
                vals_a = df_antes[col].dropna().values
                vals_d = df_depois[col].dropna().values

                ax_a.hist(vals_a, bins=50, color="#e74c3c", alpha=0.75, edgecolor="none")
                ax_a.set_title(f"{col}\n[ANTES]", fontsize=7, pad=2)
                ax_a.tick_params(labelsize=6)

                ax_d.hist(vals_d, bins=50, color="#2ecc71", alpha=0.75, edgecolor="none")
                ax_d.set_title(f"{col}\n[DEPOIS]", fontsize=7, pad=2)
                ax_d.tick_params(labelsize=6)

            fig.suptitle(
                f"Distribuição Antes × Depois — RobustScaler\n"
                f"{cenario} | Estratégia: {strategy} | Top-{top_n} features",
                fontsize=10, y=1.01,
            )
            out = fig_dir / f"distribuicao_antes_depois_top{top_n}_{strategy}.png"
            fig.savefig(out, dpi=dpi, bbox_inches="tight")
            plt.close(fig)
            console.print(f"   [success]🖼  {out.name}[/]")
        except Exception as e:
            console.print(f"   [warning]⚠  Figura distribuição falhou: {e}[/]")

        # ── 2. Boxplot features normalizadas ──────────────────────────────────
        # Linhas de referência em 0 (mediana esperada) e ±1 (IQR esperado)
        try:
            vals_box = [df_depois[c].dropna().values for c in cols]
            fig, ax  = plt.subplots(figsize=(max(10, n * 0.7), 6))
            ax.boxplot(
                vals_box, labels=cols,
                patch_artist=True, notch=False,
                medianprops=dict(color="#e74c3c", linewidth=1.5),
                boxprops=dict(facecolor="#3498db", alpha=0.6),
                whiskerprops=dict(color="#2c3e50"),
                flierprops=dict(marker=".", markersize=2, alpha=0.3, color="#e74c3c"),
            )
            ax.axhline(0,  color="#27ae60", linestyle="--", linewidth=0.8, alpha=0.7,
                       label="Mediana ref. (0)")
            ax.axhline(1,  color="#f39c12", linestyle=":",  linewidth=0.8, alpha=0.6,
                       label="IQR ref. (+1)")
            ax.axhline(-1, color="#f39c12", linestyle=":",  linewidth=0.8, alpha=0.6,
                       label="IQR ref. (−1)")
            ax.set_xticklabels(cols, rotation=60, ha="right", fontsize=7)
            ax.set_ylabel("Valor normalizado (RobustScaler)", fontsize=9)
            ax.set_title(
                f"Boxplot — Features Normalizadas\n"
                f"{cenario} | Estratégia: {strategy} | Top-{top_n}",
                fontsize=10,
            )
            ax.legend(fontsize=8)
            plt.tight_layout()
            out = fig_dir / f"boxplot_features_normalizadas_top{top_n}_{strategy}.png"
            fig.savefig(out, dpi=dpi, bbox_inches="tight")
            plt.close(fig)
            console.print(f"   [success]🖼  {out.name}[/]")
        except Exception as e:
            console.print(f"   [warning]⚠  Figura boxplot falhou: {e}[/]")

    # ──────────────────────────────────────────────────────────────────────────
    # 💾 EXPORTAÇÃO
    # ──────────────────────────────────────────────────────────────────────────

    def _export(
        self,
        result         : NormResult,
        df_treino_norm : pd.DataFrame,
        df_teste_norm  : pd.DataFrame,
        df_treino_raw  : pd.DataFrame,
        scaler         : RobustScaler,
        out_dir        : Path,
    ) -> None:
        """
        Persiste todos os artefatos do cenário + estratégia:
          Parquet treino/teste, scaler.pkl, scaler_params.json,
          CSV de auditoria e figuras de distribuição/boxplot.
        """
        out_dir.mkdir(parents=True, exist_ok=True)
        fig_dir  = out_dir / "figuras"
        cenario  = result.cenario
        strategy = result.strategy

        # ── Parquet treino ────────────────────────────────────────────────────
        with self.timer.track(f"[{cenario}/{strategy}] Salvar TREINO_NORM"):
            df_treino_norm.astype(np.float32).to_parquet(
                out_dir / "TREINO_NORM.parquet", index=False
            )
        console.print(
            f"   [success]✅ TREINO_NORM.parquet salvo "
            f"({df_treino_norm.shape[0]:,} × {df_treino_norm.shape[1]} cols)[/]"
        )

        # ── Parquet teste ─────────────────────────────────────────────────────
        with self.timer.track(f"[{cenario}/{strategy}] Salvar TESTE_NORM"):
            df_teste_norm.astype(np.float32).to_parquet(
                out_dir / "TESTE_NORM.parquet", index=False
            )
        console.print(
            f"   [success]✅ TESTE_NORM.parquet salvo "
            f"({df_teste_norm.shape[0]:,} × {df_teste_norm.shape[1]} cols)[/]"
        )

        # ── Scaler serializado ────────────────────────────────────────────────
        with open(out_dir / "scaler.pkl", "wb") as fh:
            pickle.dump(scaler, fh, protocol=pickle.HIGHEST_PROTOCOL)
        console.print(f"   [success]✅ scaler.pkl salvo[/]")

        # ── JSON de parâmetros — mediana e IQR por feature ────────────────────
        params_out = {
            "cenario"             : cenario,
            "strategy"            : strategy,
            "target_col"          : result.target_col,
            "gerado_em"           : datetime.now().isoformat(),
            "scaler_type"         : "RobustScaler",
            "quantile_range"      : list(self.cfg["quantile_range"]),
            "n_cols_normalized"   : len(result.cols_normalized),
            "n_cols_passthrough"  : len(result.cols_passthrough),
            "cols_normalized"     : result.cols_normalized,
            "cols_passthrough"    : result.cols_passthrough,
            "feature_params"      : result.scaler_params,
        }
        with open(out_dir / "scaler_params.json", "w", encoding="utf-8") as fh:
            json.dump(params_out, fh, ensure_ascii=False, indent=2)
        console.print(f"   [success]✅ scaler_params.json salvo[/]")

        # ── CSV de auditoria — estatísticas antes e depois ────────────────────
        stats_a = result.treino_stats_antes.add_suffix("_antes")
        stats_d = result.treino_stats_depois.add_suffix("_depois")
        audit   = pd.concat([stats_a, stats_d], axis=1)
        audit.to_csv(
            out_dir / "auditoria_stats_antes_depois.csv",
            sep=";", decimal=",", encoding="utf-8-sig",
        )
        console.print(f"   [success]✅ auditoria_stats_antes_depois.csv salvo[/]")

        # ── Figuras de distribuição e boxplot ─────────────────────────────────
        console.print(f"\n   [info]🖼  Gerando figuras em {fig_dir}...[/]")
        self._save_figures(
            df_treino_raw, df_treino_norm,
            result.cols_normalized, result.target_col,
            fig_dir, cenario, strategy,
        )

    # ──────────────────────────────────────────────────────────────────────────
    # 📊 DISPLAY RICH
    # ──────────────────────────────────────────────────────────────────────────

    def _print_scaler_table(self, result: NormResult, top_n: int = 20) -> None:
        """
        Exibe tabela Rich com os parâmetros do RobustScaler (mediana e IQR)
        para as top-N features normalizadas e todas as passthrough.
        """
        console.print()
        console.print(Panel(
            f"[bold white]📐 PARÂMETROS ROBUSTSCALER — "
            f"[cyan]{result.cenario}[/cyan] | [cyan]{result.strategy}[/cyan][/bold white]",
            border_style="cyan", padding=(0, 2),
        ))
        t = Table(
            box=box.ROUNDED, show_header=True,
            header_style="bold white on dark_blue",
            show_lines=False, padding=(0, 1),
        )
        t.add_column("#",            style="muted",  justify="right",  width=4)
        t.add_column("Feature",      style="bold",   justify="left",   min_width=40)
        t.add_column("Mediana",      justify="right", width=12)
        t.add_column("IQR (scale_)", justify="right", width=14)
        t.add_column("Tipo",         justify="center", width=12)

        all_cols = result.cols_normalized + result.cols_passthrough
        for rank, col in enumerate(all_cols[: top_n + len(result.cols_passthrough)], start=1):
            if col in result.scaler_params:
                p       = result.scaler_params[col]
                med_str = f"{p['center_']:.4f}"
                iqr_str = f"{p['scale_']:.4f}"
                tipo    = "[norm]NORM[/norm]"
            else:
                med_str = "—"
                iqr_str = "—"
                tipo    = "[passthru]PASS[/passthru]"
            t.add_row(str(rank), col, med_str, iqr_str, tipo)
        console.print(t)

    def _print_summary_panel(self, result: NormResult) -> None:
        """Exibe painel Rich com resumo do resultado do cenário + estratégia."""
        t = Table(
            box=box.SIMPLE_HEAVY, show_header=True,
            header_style="info", padding=(0, 2),
        )
        t.add_column("Métrica", style="muted",   width=36)
        t.add_column("Valor",   style="success", justify="right", width=18)
        t.add_row("Cenário",                          result.cenario)
        t.add_row("Linhas treino",                    f"{result.n_treino_rows:,}")
        t.add_row("Linhas teste",                     f"{result.n_teste_rows:,}")
        t.add_row("[norm]Colunas normalizadas[/norm]",
                  f"[norm]{len(result.cols_normalized)}[/norm]")
        t.add_row("[passthru]Colunas passthrough[/passthru]",
                  f"[passthru]{len(result.cols_passthrough)}[/passthru]")
        t.add_row("TARGET (sem normalização)", result.target_col)
        t.add_row("Scaler",                   "RobustScaler")
        t.add_row("Quantile range",           str(self.cfg["quantile_range"]))
        console.print(Panel(
            t,
            title=f"[header] 📋 RESUMO — {result.cenario} | {result.strategy} [/header]",
            border_style="green", padding=(0, 2),
        ))

    # ──────────────────────────────────────────────────────────────────────────
    # 🚀 PIPELINE POR CENÁRIO + ESTRATÉGIA
    # ──────────────────────────────────────────────────────────────────────────

    def _run_strategy(self, cenario_label: str, strategy: str) -> Optional[Dict]:
        """
        Executa o pipeline completo para uma combinação cenário + estratégia:
          leitura → detecção de target → monitoramento de RAM →
          estatísticas antes → fit/transform → estatísticas depois →
          display → exportação → liberação de memória.
        Retorna dicionário de resultado ou dict de erro se o arquivo não existir.
        """
        console.print()
        console.rule(f"[header] {cenario_label} | NORMALIZAÇÃO: {strategy} [/header]")

        treino_in = self.input_dir / cenario_label / strategy / "TREINO_SELECTED.parquet"
        teste_in  = self.input_dir / cenario_label / strategy / "TESTE_SELECTED.parquet"
        out_dir   = self.save_dir  / cenario_label / strategy

        if not treino_in.exists():
            console.print(f"   [error]✘  TREINO não encontrado: {treino_in}[/]")
            return {
                "cenario" : cenario_label,
                "strategy": strategy,
                "status"  : "ERRO (arquivo não encontrado)",
            }

        t0 = time.perf_counter()

        # ── Leitura dos conjuntos de treino e teste ───────────────────────────
        with self.timer.track(f"[{cenario_label}/{strategy}] Leitura TREINO"):
            df_treino = self._load_parquet(treino_in, "TREINO")
        n_treino = len(df_treino)

        df_teste = None
        n_teste  = 0
        if teste_in.exists():
            with self.timer.track(f"[{cenario_label}/{strategy}] Leitura TESTE"):
                df_teste = self._load_parquet(teste_in, "TESTE")
            n_teste = len(df_teste)
        else:
            console.print(f"   [warning]⚠  TESTE não encontrado — continuando apenas com treino[/]")
            df_teste = pd.DataFrame(columns=df_treino.columns)

        # ── Detecção do TARGET e monitoramento de RAM ─────────────────────────
        console.print("\n   [info]🎯 Detectando TARGET...[/]")
        target_col = detectar_target(df_treino)
        self._print_ram("pré-normalização")

        # ── Estatísticas antes (limitado a 50 colunas para performance) ───────
        all_num_cols   = df_treino.select_dtypes(include=[np.number]).columns.tolist()
        cols_para_desc = [c for c in all_num_cols if c != target_col]
        stats_antes    = self._describe_cols(df_treino, cols_para_desc[:50])

        # ── Fit + Transform com garantia anti-leakage ─────────────────────────
        console.print("\n   [info]⚙️  Iniciando fit/transform RobustScaler...[/]")
        (
            df_treino_norm, df_teste_norm, scaler,
            cols_normalized, cols_passthrough, scaler_params,
        ) = self._fit_transform(df_treino, df_teste, target_col, cenario_label, strategy)

        self._print_ram("pós-normalização")

        # ── Estatísticas depois ───────────────────────────────────────────────
        cols_desc_depois = [c for c in cols_normalized if c in df_treino_norm.columns][:50]
        stats_depois     = self._describe_cols(df_treino_norm, cols_desc_depois)

        result = NormResult(
            cenario             = cenario_label,
            strategy            = strategy,
            target_col          = target_col,
            cols_normalized     = cols_normalized,
            cols_passthrough    = cols_passthrough,
            n_treino_rows       = n_treino,
            n_teste_rows        = n_teste,
            scaler_params       = scaler_params,
            treino_stats_antes  = stats_antes,
            treino_stats_depois = stats_depois,
        )

        self._print_summary_panel(result)
        self._print_scaler_table(result, top_n=20)

        # ── Exportação de todos os artefatos ──────────────────────────────────
        console.print(Rule("[info]💾  EXPORTAÇÃO[/info]"))
        with self.timer.track(f"[{cenario_label}/{strategy}] Exportação"):
            self._export(
                result, df_treino_norm, df_teste_norm,
                df_treino, scaler, out_dir,
            )

        self._force_free(df_treino, df_teste, df_treino_norm, df_teste_norm)
        elapsed = time.perf_counter() - t0
        self._print_ram("após liberação")

        return {
            "cenario"            : cenario_label,
            "strategy"           : strategy,
            "target_col"         : target_col,
            "n_cols_normalized"  : len(cols_normalized),
            "n_cols_passthrough" : len(cols_passthrough),
            "n_treino_rows"      : n_treino,
            "n_teste_rows"       : n_teste,
            "elapsed_s"          : round(elapsed, 2),
            "status"             : "OK",
        }

    # ──────────────────────────────────────────────────────────────────────────
    # 📋 RELATÓRIOS FINAIS
    # ──────────────────────────────────────────────────────────────────────────

    def _print_time_report(self, elapsed_total: float) -> None:
        """Exibe tabela Rich com o tempo de cada estágio e o total da execução."""
        console.print()
        console.print(Rule("[success]⏱️  RELATÓRIO DE TEMPO[/success]"))
        t = Table(box=box.SIMPLE_HEAVY, show_header=True, header_style="info")
        t.add_column("Estágio",   style="muted",   min_width=50)
        t.add_column("Tempo (s)", style="success", justify="right", width=12)
        for label, elapsed in self.timer.records.items():
            t.add_row(label, f"{elapsed:.2f}s")
        t.add_row("[bold]TOTAL[/bold]", f"[bold]{elapsed_total:.2f}s[/bold]")
        console.print(t)

    def _print_final_table(self) -> None:
        """Exibe tabela consolidada com todos os cenários e estratégias processados."""
        t = Table(
            title="RELATÓRIO CONSOLIDADO — NORMALIZAÇÃO | PLD NORDESTE | CENÁRIO A",
            border_style="blue", box=box.DOUBLE_EDGE,
            show_lines=True, min_width=120,
        )
        t.add_column("Cenário",       style="bold",      justify="left",  min_width=28)
        t.add_column("Estratégia",    style="bold cyan", justify="left",  min_width=20)
        t.add_column("TARGET",        style="dim",       justify="left",  min_width=28)
        t.add_column("Cols Norm",     style="norm",      justify="right", min_width=10)
        t.add_column("Cols Pass",     style="passthru",  justify="right", min_width=10)
        t.add_column("Linhas Treino", justify="right",   min_width=14)
        t.add_column("Linhas Teste",  justify="right",   min_width=13)
        t.add_column("Tempo (s)",     style="white",     justify="right", min_width=10)
        t.add_column("Status",        style="bold",      justify="center", min_width=8)

        for r in self._all_results:
            is_a   = "todos" in r.get("cenario", "")
            cor_c  = "cenario_a" if is_a else "cenario_b"
            status = "[green]OK[/]" if r.get("status") == "OK" \
                else f"[red]{r.get('status')}[/]"
            t.add_row(
                f"[{cor_c}]{r.get('cenario', '—')}[/{cor_c}]",
                r.get("strategy",           "—"),
                r.get("target_col",         "—"),
                str(r.get("n_cols_normalized",  0)),
                str(r.get("n_cols_passthrough", 0)),
                f"{r.get('n_treino_rows', 0):,}",
                f"{r.get('n_teste_rows',  0):,}",
                str(r.get("elapsed_s",      0)),
                status,
            )
        console.print(t)

    def _save_consolidated_report(self) -> None:
        """Persiste o relatório consolidado de todos os cenários em CSV."""
        out = self.save_dir / "relatorio_normalizacao_nordeste.csv"
        out.parent.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(self._all_results).to_csv(out, index=False, encoding="utf-8-sig")
        console.print(f"   [success]✔  Relatório consolidado: {out}[/]")

    # ──────────────────────────────────────────────────────────────────────────
    # 🚀 ORQUESTRADOR PRINCIPAL
    # ──────────────────────────────────────────────────────────────────────────

    def run(self) -> None:
        """
        Ponto de entrada do pipeline. Exibe painéis de boas-vindas e
        arquitetura anti-leakage, itera sobre cenários e estratégias,
        imprime relatórios finais e persiste o CSV consolidado.
        """
        console.print()
        console.print(Panel(
            Text.assemble(
                ("📐 NORMALIZAÇÃO DE DADOS — PLD NORDESTE\n", "bold green"),
                ("   Cenário A (todos os anos)\n", "muted"),
                (f"   Input  : {self.input_dir}\n", "muted"),
                (f"   Output : {self.save_dir}\n",  "muted"),
                (f"   Run em : {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}\n", "muted"),
                ("   Scaler : ", "muted"),
                ("RobustScaler (mediana + IQR) | Anti-Leakage", "warning"),
            ),
            title="[header]  TCC PLD — PASSO 10: NORMALIZAÇÃO | NORDESTE v3.0  [/header]",
            border_style="green", padding=(1, 4),
        ))

        # ── Painel explicativo da arquitetura anti-leakage ────────────────────
        console.print(Panel(
            "[warning]GARANTIA ANTI-LEAKAGE:[/]\n\n"
            "  [bold]FIT[/]  do RobustScaler → SOMENTE no TREINO de cada cenário\n"
            "  [bold]TRANSFORM[/]            → TREINO + TESTE com os mesmos parâmetros\n"
            "  [bold]TARGET[/]               → NÃO normalizado (escala original preservada)\n"
            "  [bold]KEY_* / Binárias[/]     → NÃO normalizadas (passthrough)\n\n"
            "  [dim]O TESTE não influencia nenhuma decisão de fit.[/]\n"
            "  [dim]Cada cenário tem seu próprio scaler independente.[/]",
            title="[header] ARQUITETURA ANTI-LEAKAGE [/header]",
            border_style="yellow", padding=(0, 2),
        ))

        t_total = time.perf_counter()

        # ── Iteração sobre cenários e estratégias ─────────────────────────────
        for scenario in self.scenarios:
            cenario_label = scenario["label"]
            cor           = scenario["cor"]
            border        = scenario["border"]

            console.print()
            console.rule(
                f"[{cor}]{'═'*20}  CENÁRIO: {cenario_label}  {'═'*20}[/{cor}]"
            )
            console.print(Panel(
                f"[{cor}]{scenario['desc']}[/{cor}]\n"
                f"[muted]Input : {self.input_dir / cenario_label}[/]\n"
                f"[muted]Output: {self.save_dir  / cenario_label}[/]",
                border_style=border,
                padding=(0, 2),
            ))

            for strategy in self.strategies:
                row = self._run_strategy(cenario_label, strategy)
                if row:
                    self._all_results.append(row)

        # ── Relatórios e painel de encerramento ───────────────────────────────
        elapsed_total = time.perf_counter() - t_total
        self._print_time_report(elapsed_total)
        self._print_final_table()
        self._save_consolidated_report()

        n_ok  = sum(1 for r in self._all_results if r.get("status") == "OK")
        n_err = len(self._all_results) - n_ok
        used, tot, style = self._ram_status()

        console.print()
        console.print(Panel(
            Text.assemble(
                ("✅ NORMALIZAÇÃO NORDESTE CONCLUÍDA\n", "bold green"),
                (f"   {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}\n", "muted"),
                (f"   Cenários OK  : {n_ok} | Erros: {n_err}\n", "muted"),
                (f"   Tempo total  : {elapsed_total:.2f}s\n", "muted"),
                (f"   RAM final    : {used:.1f}/{tot:.1f} GB", style),
            ),
            border_style="green", padding=(1, 4),
        ))


# ==============================================================================
# ▶️ ENTRY POINT
# ==============================================================================
if __name__ == "__main__":
    engine = NormalizationEngine(
        input_dir  = INPUT_DIR,
        save_dir   = SAVE_DIR,
        scenarios  = SCENARIOS,
        strategies = STRATEGIES,
        config     = NORM_CONFIG,
    )
    engine.run()